In [7]:
pip install biopython

Note: you may need to restart the kernel to use updated packages.


Modulos Necessarios

In [8]:
from Bio import SeqIO, SwissProt
from Bio.SeqRecord import SeqRecord
from io import StringIO
import requests

Analise de domínios conservados (CDD) da proteina

In [9]:
uniprot_ids = ["A0A125SA63", "A0A125SA64", "A0A193DRX6"]

for uniprot_id in uniprot_ids:
    fasta_url = f"https://www.uniprot.org/uniprotkb/{uniprot_id}.fasta?format=fasta"
    r = requests.get(fasta_url)

    if r.status_code == 200:
        fasta_file = f"{uniprot_id}.fasta"
        with open(fasta_file, "w") as f:
            f.write(r.text)

        record = SeqIO.read(fasta_file, "fasta")
        protein_seq = str(record.seq)
        print(f"ID: {record.id}")
        print(f"Comprimento: {len(protein_seq)} aa")
        print("="*40)


    txt_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.txt"
    r = requests.get(txt_url)

    record_sp = SwissProt.read(StringIO(r.text))

    print("Entry Name:", record_sp.entry_name)
    print("Organismo:", record_sp.organism)
    print("Comprimento:", record_sp.sequence_length)
    print("Curation Status:", record_sp.data_class)

    print("\nDescrição/Função:")
    for desc in record_sp.description.split(";"):
        print("-", desc.strip())

ID: tr|A0A125SA63|A0A125SA63_9CAUD
Comprimento: 166 aa
Entry Name: A0A125SA63_9CAUD
Organismo: Pseudomonas phage AAT-1.
Comprimento: 166
Curation Status: Unreviewed

Descrição/Função:
- SubName: Full=Endolysin {ECO:0000313|EMBL:AME18059.1}
- 
ID: tr|A0A125SA64|A0A125SA64_9CAUD
Comprimento: 161 aa
Entry Name: A0A125SA64_9CAUD
Organismo: Pseudomonas phage AAT-1.
Comprimento: 161
Curation Status: Unreviewed

Descrição/Função:
- SubName: Full=Putative i-spanin {ECO:0000313|EMBL:AME18060.1}
- 
ID: tr|A0A193DRX6|A0A193DRX6_9CAUD
Comprimento: 120 aa
Entry Name: A0A193DRX6_9CAUD
Organismo: Pseudomonas phage AAT-1.
Comprimento: 120
Curation Status: Unreviewed

Descrição/Função:
- SubName: Full=Putative o-spanin {ECO:0000313|EMBL:ANN44564.1}
- 


Extrair domínios conservados via CDD

In [10]:
genes = [
    {"locus": "P9A56_gp33", "name": "endolysin", "fasta": "P9A56_gp33_protein.fasta"},
    {"locus": "P9A56_gp34", "name": "putative i-spanin", "fasta": "P9A56_gp34_protein.fasta"},
    {"locus": "P9A56_gp35", "name": "putative o-spanin", "fasta": "P9A56_gp35_protein.fasta"}
]

for gene in genes:
    protein_record = SeqIO.read(gene["fasta"], "fasta")
    protein_seq_str = str(protein_record.seq)

    url = "https://www.ncbi.nlm.nih.gov/Structure/bwrpsb/bwrpsb.cgi"
    params = {
        "seq": protein_seq_str,
        "db": "cdd",
        "smode": "auto",
        "output": "text"
    }

    r = requests.post(url, data=params)
    if r.status_code == 200:
        with open(f"{gene['locus']}_cdd.txt", "w", encoding="utf-8") as f:
            f.write(r.text)



Extração das Informações Relevantes de CDD

In [11]:
genes = [
    {"locus": "P9A56_gp33"},
    {"locus": "P9A56_gp34"},
    {"locus": "P9A56_gp35"}
]

for gene in genes:
    cdd_file = f"{gene['locus']}_cdd.txt"
    print(f"\n\n=== Domínios Conservados (CDD) para {gene['locus']} ===\n")

    with open(cdd_file, encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        if "cd" in line.lower() and "e-value" in line.lower():
            print(line.strip())




=== Domínios Conservados (CDD) para P9A56_gp33 ===

'content': 'The Full display <b>shows all domain models</b>, as available for each region on the query sequence, that meet or exceed the RPS-BLAST threshold for statistical significance (i.e., the <a href="/Structure/cdd/cdd_help.shtml#WRPSBExpect">E-value cutoff</a>).  The <a href="/Structure/cdd/cdd_help.shtml#RPSB_hit_types">hit types</a> can include <a href="/Structure/cdd/cdd_help.shtml#RPSB_hit_type_specific_hit">specific hits</a>, <a href="/Structure/cdd/cdd_help.shtml#RPSB_hit_type_non_specific_hit">non-specific hits</a>, the <a href="/Structure/cdd/cdd_help.shtml#RPSB_hit_type_superfamily">superfamily(ies)</a> to which those hits belong, and <a href="/Structure/cdd/cdd_help.shtml#RPSB_hit_type_multi_domain">multi-domain models</a>.<br>The bottom of the Full Display also summarizes <b>BLAST search parameters</b>, which include a summary of information such as the <a href="/Structure/cdd/cdd_help.shtml#RPSBSearchDb">database 